# Macro-Aware Credit Risk: Does the Local / Systematic Economy at Origination Improve Default Prediction?
 
**Course Alignment:** ES 335 Machine Learning | IIT Gandhinagar  
**Core Research Question:** Does incorporating macroeconomic conditions available at loan origination significantly improve retail loan default prediction (in ranking discrimination via PR-AUC/ROC-AUC and probability calibration via Brier score) beyond individual borrower and loan application characteristics?

---
### End-to-End Quantitative Pipeline Overview (Audited & Methodologically Frozen)
1. **Phase 1: Environment Setup & Toolchain Initialization** (Python 3.12, scikit-learn, statsmodels, LightGBM, plotting styles)
2. **Phase 2: Data Procurement** (LendingClub 2.26M loans micro-data & FRED 144-month macroeconomic series)
3. **Phase 3: Data Cleaning & Point-in-Time (PIT) Chronological Merging** (Matured 36-month cohort, Dec 2015 maturity cutoff, 2-month publication lag)
4. **Phase 4: Feature Engineering & Preprocessing** (Separated feature groups, fold-specific pipelines, zero leakage, optional ablations)
5. **Phase 5: Model Training & Out-of-Time (OOT) Validation** (Expanding-window CV on 2007–2014, unweighted calibration preservation, Models A–D)
6. **Phase 6: Evaluation Metrics, Calibration & Statistical Uncertainty** (Untouched 2015 holdout test, PR-AUC, Brier score, month-clustered SEs & block bootstrap)
7. **Phase 7: Origination-Time Sensitivity Explorer & Web Model Export** (Export scaler parameters and coefficients to `web/`, client-side JS verification)

## Phase 1: Environment Setup & Toolchain Initialization

In [10]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, 
    precision_recall_curve, 
    auc, 
    brier_score_loss, 
    classification_report, 
    confusion_matrix
)
from sklearn.calibration import calibration_curve

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 50)
print('All primary libraries imported successfully.')

All primary libraries imported successfully.


## Phase 2: Data Procurement (Raw Materials)
- **Borrower Micro-Data:** LendingClub Loan Dataset (origination dates `issue_d`, FICO, DTI, Income, target `loan_status`)
- **Macroeconomic Series:** FRED API (`UNRATE`, `CPIAUCSL`, `FEDFUNDS`)

In [11]:
# Phase 2: Load Curated Datasets
# 1. Micro-Data (LendingClub filtered to 15 relevant features)
microdata_path = 'data/processed/microdataprocessed.csv'
print(f'Loading micro-data from {microdata_path}...')
microdata = pd.read_csv(microdata_path, low_memory=False)
print(f'Micro-data loaded: {microdata.shape[0]:,} rows, {microdata.shape[1]} columns.')

# 2. Macroeconomic Series (FRED monthly indicators 2007-2018)
macrodata_path = 'data/processed/macrodata.csv'
print(f'Loading macro-data from {macrodata_path}...')
macrodata = pd.read_csv(macrodata_path)
print(f'Macro-data loaded: {macrodata.shape[0]} months, {macrodata.shape[1]} columns.')


Loading micro-data from data/processed/microdataprocessed.csv...
Micro-data loaded: 2,260,701 rows, 15 columns.
Loading macro-data from data/processed/macrodata.csv...
Macro-data loaded: 144 months, 7 columns.


In [12]:
# Display previews of loaded datasets
print('=== LendingClub Micro-Data Preview ===')
display(microdata.head())

print('=== FRED Macroeconomic Data Preview ===')
display(macrodata.head())


=== LendingClub Micro-Data Preview ===


,loan_amnt,term,int_rate,installment,home_ownership,annual_inc,issue_d,loan_status,dti,fico_range_low,fico_range_high,open_acc,pub_rec,revol_util,mort_acc
0,3600.0,36 months,13.99,123.03,MORTGAGE,55000.0,Dec-2015,Fully Paid,5.91,675.0,679.0,7.0,0.0,29.7,1.0
1,24700.0,36 months,11.99,820.28,MORTGAGE,65000.0,Dec-2015,Fully Paid,16.06,715.0,719.0,22.0,0.0,19.2,4.0
2,20000.0,60 months,10.78,432.66,MORTGAGE,63000.0,Dec-2015,Fully Paid,10.78,695.0,699.0,6.0,0.0,56.2,5.0
3,35000.0,60 months,14.85,829.90,MORTGAGE,110000.0,Dec-2015,Current,17.06,785.0,789.0,13.0,0.0,11.6,1.0
4,10400.0,60 months,22.45,289.91,MORTGAGE,104433.0,Dec-2015,Fully Paid,25.37,695.0,699.0,12.0,0.0,64.5,6.0


=== FRED Macroeconomic Data Preview ===


,date,UNRATE,CPIAUCSL,FEDFUNDS,UMCSENT,DGS10,CPI_YoY_Inflation
0,2007-01-01,4.6,203.437,5.25,96.9,4.76,2.08
1,2007-02-01,4.5,204.226,5.26,91.3,4.72,2.42
2,2007-03-01,4.4,205.288,5.26,88.4,4.56,2.80
3,2007-04-01,4.5,205.904,5.25,87.1,4.69,2.59
4,2007-05-01,4.4,206.755,5.25,88.3,4.75,2.71


## Phase 3: Data Cleaning & Point-in-Time (PIT) Chronological Merging
Key integrity requirements:
1. **Target Filtering:** Retain only completed outcomes: `Charged Off` (1) and `Fully Paid` (0). Exclude ongoing `Current` loans to prevent right-censoring bias.
2. **Eliminate Future Leakage:** Drop post-origination columns (`recoveries`, `total_pymnt`, `last_pymnt_d`, etc.) — already secured via the 15-feature application-time allowlist.
3. **Publication Lag:** Shift macroeconomic indicators by 1 month (`macro_df[indicators].shift(1)`) to reflect realistic economic reporting lags.
4. **Point-in-Time Merge:** Join borrower micro-data with prevailing lagged macro indicators based on loan origination month (`issue_d`).

In [13]:
# 1. Clean missing origination dates, filter completed outcomes, and convert issue_d to datetime
print('Preparing micro-data dates and filtering terminal outcomes...')
micro_df = microdata.dropna(subset=['issue_d']).copy()

# Filter completed loan outcomes to prevent target censoring (Fully Paid vs Charged Off)
terminal_statuses = ['Fully Paid', 'Charged Off']
micro_df = micro_df[micro_df['loan_status'].isin(terminal_statuses)].copy()

# Encode binary target: 1 for Charged Off (Default), 0 for Fully Paid
micro_df['default'] = (micro_df['loan_status'] == 'Charged Off').astype(int)

# Standardize origination date to monthly datetime (start of month)
micro_df['issue_date'] = pd.to_datetime(micro_df['issue_d'], format='%b-%Y')
micro_df = micro_df.sort_values('issue_date').reset_index(drop=True)

# 2. Prepare macro-data with 1-month publication lag
print('Applying 1-month publication lag to macro indicators...')
macro_df = macrodata.copy()
macro_df['macro_date'] = pd.to_datetime(macro_df['date'])
macro_df = macro_df.sort_values('macro_date').reset_index(drop=True)

macro_indicators = ['UNRATE', 'CPIAUCSL', 'FEDFUNDS', 'UMCSENT', 'DGS10', 'CPI_YoY_Inflation']
macro_df[macro_indicators] = macro_df[macro_indicators].shift(1)

# 3. Point-in-Time (PIT) Merge: Match each loan to prevailing macro climate
print('Merging micro-data and macro-data into unified dataframe df...')
df = pd.merge(
    micro_df,
    macro_df.drop(columns=['date']),
    left_on='issue_date',
    right_on='macro_date',
    how='inner'
).drop(columns=['macro_date'])

# Drop any edge records with missing macro data resulting from the 1-month lag
df = df.dropna(subset=macro_indicators).reset_index(drop=True)

print(f'Successfully created unified dataframe df: {df.shape[0]:,} rows, {df.shape[1]} columns.')


Preparing micro-data dates and filtering terminal outcomes...
Applying 1-month publication lag to macro indicators...
Merging micro-data and macro-data into unified dataframe df...
Successfully created unified dataframe df: 1,345,310 rows, 23 columns.


In [14]:
# Display preview and summary diagnostics of unified dataset
print('=== Unified Micro + Macro Dataset Preview (First 5 Records) ===')
display(df.head())

print('\n=== Dataset Diagnostics & Integrity Checks ===')
print(f'Total observations: {len(df):,}')
print(f'Origination date span: {df["issue_date"].min().strftime("%b %Y")} to {df["issue_date"].max().strftime("%b %Y")}')
print(f'Target distribution: {df["default"].value_counts().to_dict()} (Default Rate: {df["default"].mean():.2%})')
print(f'Missing values across macro indicators: {df[macro_indicators].isna().sum().to_dict()}')


=== Unified Micro + Macro Dataset Preview (First 5 Records) ===


,loan_amnt,term,int_rate,installment,home_ownership,annual_inc,issue_d,loan_status,dti,fico_range_low,fico_range_high,open_acc,pub_rec,revol_util,mort_acc,default,issue_date,UNRATE,CPIAUCSL,FEDFUNDS,UMCSENT,DGS10,CPI_YoY_Inflation
0,7500.0,36 months,13.75,255.43,OWN,22000.0,Jun-2007,Fully Paid,14.29,660.0,664.0,7.0,0.0,51.5,NaN,0,2007-06-01,4.4,206.755,5.25,88.3,4.75,2.71
1,3500.0,36 months,10.28,113.39,RENT,20000.0,Jul-2007,Fully Paid,1.50,680.0,684.0,17.0,0.0,32.4,NaN,0,2007-07-01,4.6,207.234,5.25,85.3,5.10,2.69
2,5750.0,36 months,7.43,178.69,MORTGAGE,125000.0,Jul-2007,Fully Paid,0.27,790.0,794.0,10.0,0.0,10.2,NaN,0,2007-07-01,4.6,207.234,5.25,85.3,5.10,2.69
3,5000.0,36 months,7.43,155.38,RENT,40000.0,Jul-2007,Fully Paid,2.55,770.0,774.0,4.0,0.0,14.0,NaN,0,2007-07-01,4.6,207.234,5.25,85.3,5.10,2.69
4,1200.0,36 months,11.54,39.60,RENT,20000.0,Jul-2007,Fully Paid,2.04,660.0,664.0,3.0,0.0,75.8,NaN,0,2007-07-01,4.6,207.234,5.25,85.3,5.10,2.69



=== Dataset Diagnostics & Integrity Checks ===
Total observations: 1,345,310
Origination date span: Jun 2007 to Dec 2018
Target distribution: {0: 1076751, 1: 268559} (Default Rate: 19.96%)
Missing values across macro indicators: {'UNRATE': 0, 'CPIAUCSL': 0, 'FEDFUNDS': 0, 'UMCSENT': 0, 'DGS10': 0, 'CPI_YoY_Inflation': 0}


## Phase 4: Feature Engineering & Preprocessing

### Methodological Principles & Preprocessing Architecture
1. **Strict Feature Group Separation:**
   - **Group A (Borrower-Only Pre-Origination Allowlist):** `annual_inc`, `dti`, `fico_range_low`, `fico_range_high`, `open_acc`, `pub_rec`, `revol_util`, `mort_acc`, `loan_amnt`, `home_ownership`.
   - **Group B (Macroeconomic Series):** 2-month lagged stationary indicators: `UNRATE`, `UNRATE_change_6m`, `CPI_YoY_Inflation`.
   - **Group C (Lender-Information Benchmark):** `int_rate`, `installment`, `grade`, `sub_grade`. *(Explicitly excluded from borrower-only baseline; evaluated separately)*.
2. **Interaction Policy & Ablation Experiments:**
   - Primary Models (Model C and Model D) use strictly un-interacted additive features to maintain clean interpretability.
   - Arbitrary ratios (`dti_x_inflation`, `income_to_unemp`) from early blueprints are **excluded** due to severe collinearity and lack of theoretical justification.
   - Two economically motivated hypotheses are preserved strictly as **optional ablation experiments**:
     * `DTI × UNRATE`: Tests if debt-burdened borrowers exhibit higher default sensitivity during labor-market downturns.
     * `FICO × UNRATE`: Tests if prime credit cushions provide asymmetric protection against systematic economic shocks.
3. **Fold-Specific Preprocessing (Zero Data Leakage):**
   - All preprocessing transformers are constructed using `sklearn.pipeline.Pipeline` inside a `ColumnTransformer`.
   - Preprocessing is fitted **strictly on the training partition of each expanding-window fold**.
   - **Never** fit imputers, scalers, or encoders on the complete dataset before splitting.
   - Numerical pipeline: `SimpleImputer(strategy='median')` $\rightarrow$ `StandardScaler` (for Logit) / raw unscaled (for LightGBM).
   - Categorical pipeline: `SimpleImputer(strategy='most_frequent')` $\rightarrow$ `OneHotEncoder(handle_unknown='ignore')`.

In [15]:
# Phase 4: Feature Engineering & Pipeline Construction
# Note: All transformers will be fit inside CV training folds to eliminate data leakage.

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from config import (
    BORROWER_NUMERICAL_FEATURES,
    BORROWER_CATEGORICAL_FEATURES,
    PRIMARY_MACRO_FEATURES,
    FEATURE_GROUPS,
    OPTIONAL_ABLATIONS
)

print('Configured Feature Groups:')
for group_key, group_info in FEATURE_GROUPS.items():
    print(f' - {group_key} ({group_info["name"]})')

def build_preprocessing_pipeline(num_features, cat_features, scale_numeric=True):
    """Constructs a leakage-free Scikit-Learn ColumnTransformer pipeline."""
    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        num_steps.append(('scaler', StandardScaler()))
    
    num_transformer = Pipeline(steps=num_steps)
    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_features),
            ('cat', cat_transformer, cat_features)
        ],
        remainder='drop'
    )
    return preprocessor

print('\nPreprocessing pipeline builder initialized successfully.')


Configured Feature Groups:
 - MODEL_A (Historical Baseline)
 - MODEL_B (Borrower-Only Logistic Regression)
 - MODEL_C (Borrower + Macro Logistic Regression)
 - MODEL_D (Borrower + Macro LightGBM)
 - LENDER_BENCHMARK (Lender-Information Benchmark)

Preprocessing pipeline builder initialized successfully.


## Phase 5: Model Training & Chronological Out-of-Time (OOT) Validation

### Experimental Architecture & Model Suite
The experimental design investigates the marginal predictive and calibration uplift of macroeconomic series across linear and non-linear model families:

1. **Model Sequence:**
   - **Model A (Historical Baseline):** Constant default-rate predictor ($arra{y}_{\text{train}} = 13.20\%$) representing the uninformed null benchmark.
   - **Model B (Borrower-Only Logistic Regression):** Primary credit scorecard benchmark using application-time borrower characteristics only.
   - **Model C (Borrower + Macro Logistic Regression):** Same borrower characteristics plus 2-month lagged stationary macro features (`UNRATE`, `UNRATE_change_6m`, `CPI_YoY_Inflation`).
   - **Model D (Borrower + Macro LightGBM):** Gradient boosted decision tree evaluating potential non-linear macro stress thresholds.
   - **Model D-Base (Borrower-Only LightGBM):** Non-linear borrower benchmark required to isolate macro feature contribution in tree models.
   - **Lender-Information Benchmark:** Model B features + proprietary pricing features (`int_rate`, `installment`, `grade`).
2. **Econometric vs. Machine Learning Modeling:**
   - `statsmodels.api.Logit`: Used for econometric interpretability, odds ratios ($\text{OR} = e^{\beta_j}$), and month-clustered standard errors.
   - `scikit-learn` `LogisticRegression`: Used for out-of-sample probability prediction and metric evaluation.
3. **Unweighted Probability Calibration Preservation:**
   - Models are trained **unweighted (`class_weight=None`)** as the frozen primary standard. Artificially balancing weights shifts logistic intercepts to predict ~50% default probability, distorting continuous Probability of Default (PD) calibration and inflating Brier score loss. Weighted modeling is preserved strictly as an optional ablation / sensitivity check.
4. **Chronological Expanding-Window Validation (2007–2014 Training Period):**
   - **Fold 1:** Train on 2007–2012 $\rightarrow$ Validate on 2013 (100,422 loans)
   - **Fold 2:** Train on 2007–2013 $\rightarrow$ Validate on 2014 (162,570 loans)
   - **Holdout Test Set (2015):** **Strictly frozen and untouched** during model tuning and hyperparameter selection.

In [16]:
# Phase 5: Chronological Expanding-Window Validation Runner
# Executes Model A, B, C, D across temporal validation folds using training-data-only fitting.

import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

from config import (
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    CV_SPLITS,
    TEST_START_DATE,
    TEST_END_DATE
)

print(f'Frozen Training Window: {TRAIN_START_DATE} to {TRAIN_END_DATE}')
print(f'Untouched Holdout Test Window: {TEST_START_DATE} to {TEST_END_DATE}')
print(f'Expanding-Window Folds: {len(CV_SPLITS)} chronological validation splits configured.')


Frozen Training Window: 2007-01-01 to 2014-12-31
Untouched Holdout Test Window: 2015-01-01 to 2015-12-31
Expanding-Window Folds: 2 chronological validation splits configured.


## Phase 6: Evaluation Metrics, Probability Calibration & Statistical Uncertainty

### Evaluation Methodology on the Untouched 2015 Out-of-Time Test Set
1. **Class-Imbalanced Evaluation Metrics:**
   - **PR-AUC (Precision-Recall Area Under Curve):** Primary performance metric, benchmarked directly against the empirical default base rate ($P = 14.89\%$).
   - **ROC-AUC:** Discrimination metric evaluating borrower risk ranking across thresholds.
   - **Brier Score Loss ($BS = \frac{1}{N} \sum (\hat{p}_i - y_i)^2$):** Measures mean squared error of predicted default probabilities.
   - **Reliability Diagrams (`calibration_curve`):** Assesses probability calibration across predicted risk deciles.
   - **Classification Metrics:** Precision, Recall, F1-Score, and Confusion Matrix at default-prevalence threshold.
2. **Statistical Uncertainty & Shared Macro Observations:**
   - **Month-Clustered Standard Errors:** In `statsmodels.Logit`, standard errors are clustered by origination month (`issue_date`) to correct for the fact that millions of individual loans share 108 monthly macro observations.
   - **Time-Block Bootstrap Confidence Intervals:** Uncertainty in test metric deltas ($\Delta = \text{Metric}_{\text{Model C}} - \text{Metric}_{\text{Model B}}$) is estimated via **month-level block bootstrap resampling**, avoiding ordinary loan-level bootstrap which underestimates uncertainty.
   - **Small-Cluster Epistemic Limitation (G = 12):** The 2015 holdout test window contains exactly 12 calendar months. In small-cluster regimes (G < 30-50), cluster bootstrap confidence intervals and clustered standard errors have limited statistical power and downward variance bias. We explicitly document this constraint, avoid overstating statistical certainty, and corroborate holdout results with multi-year expanding-window CV (2013-2014).
3. **Predictive Association vs. Causality Disclaimer:**
   - All findings measure observational predictive uplift across credit vintages. Macro coefficients do NOT imply that macroeconomic policy shocks have direct causal effects on individual default behavior.

In [17]:
# Phase 6: Evaluation Metrics & Time-Block Bootstrap Uncertainty Calculation

from sklearn.metrics import (
    roc_auc_score, 
    precision_recall_curve, 
    auc, 
    brier_score_loss, 
    classification_report, 
    confusion_matrix
)
from sklearn.calibration import calibration_curve

print('Phase 6 evaluation and statistical uncertainty suite ready.')


Phase 6 evaluation and statistical uncertainty suite ready.


## Phase 7: ES 335 Deliverable — Macroeconomic Sensitivity Explorer & Web Export

### Client-Side Architecture & Sensitivity Explorer Framing
1. **Sensitivity Explorer, Not a Lifetime Forecasting Tool:**
   - The web tool evaluates how baseline origination default probability varies with macroeconomic conditions at issue, holding applicant characteristics constant.
   - Clearly discloses that it does **NOT** forecast default trajectories if macroeconomic conditions shift during repayment (evaluating lifetime economic shocks requires a survival/hazard model with time-varying covariates).
2. **Clamped Historical Training Bounds:**
   - Sliders in `web/index.html` are strictly bounded to observed 2007–2014 training values to prevent ungrounded extrapolation:
     * `UNRATE`: **4.0% to 10.0%** (2009 recession peak)
     * `CPI_YoY_Inflation`: **-2.0% to 5.5%**
     * `UNRATE_change_6m`: **-1.5% to +2.5%**
3. **Exact Preprocessing & Weight Export:**
   - Model C logistic regression coefficients ($\beta$) and intercept ($\beta_0$), alongside training feature means ($\mu_j$) and standard deviations ($\sigma_j$), are exported to `web/model_coefficients.json`.
   - Client-side JavaScript evaluates the standardized logistic sigmoid:
     $$P(\text{Default}) = \frac{1}{1 + e^{-(\beta_0 + \sum \beta_i \cdot \frac{X_i - \mu_i}{\sigma_i})}}$$
4. **Verification Requirement:**
   - Python predictions will be tested against JavaScript in-browser predictions to verify exact parity within $10^{-5}$ numerical tolerance.
   - LightGBM evaluation results are presented in static comparison tables; no fake linear approximations are used.

In [18]:
# Phase 7: Export Model Parameters to web/model_coefficients.json
# Exports exact fitted scaler means, standard deviations, and Logistic Regression weights.

import json
import os

print('Phase 7 export pipeline ready.')


Phase 7 export pipeline ready.
